# Predicting Abalone Age Using Neural Network #

Import all packages required

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Create read_function to read data and perform cleaning

In [2]:
def read_data():
    # Read the data file 'abalone.data' taken from https://archive.ics.uci.edu/dataset/1/abalone. 
    # Taking a look at the data file before reading shows us that the dataset contains no column name thus for the read_csv function, 
    # we must use headers=None to ensure that the first row does not become our column names.
    df = pd.read_csv('abalone.data', header = None)
    df.columns = ['Sex', 'Length', 'Diameter', 'Height','Whole_Weight', 'Shucked_Weight', 'Viscera_Weight', 'Shell_Weight', 'Rings']
    df['Ring-Age'] = df['Rings'] + 1.5
    df.drop(columns = 'Rings', inplace = True)
    # One hot encode sex
    df = pd.get_dummies(df, columns = ['Sex'], drop_first = False)
    # Convert from boolean to int
    df[['Sex_F', 'Sex_I', 'Sex_M']] = df[['Sex_F', 'Sex_I', 'Sex_M']].astype(int)
    # Classify according to bins
    bins = [-np.inf, 7, 10, 15, np.inf]
    labels = [0, 1, 2, 3]
    df['y'] = pd.cut(df['Ring-Age'], bins = bins, labels = labels).astype(int)
    df.drop(columns = 'Ring-Age', inplace = True)
    return df

Functions for feature exploration

In [3]:
features = ['Length','Diameter','Height','Whole_Weight', 'Shucked_Weight','Viscera_Weight','Shell_Weight']

def hist(df):
    for i in features:
        print(df[i].describe())
        plt.figure()
        sns.histplot(df[i], bins = 25, kde = True)
        plt.title(f'Distribution of {i}')
        plt.tight_layout()
        plt.savefig(f'{i}_dist.png')
        plt.close()

In [4]:
def boxplot(df):
    for i in features:
        plt.figure()
        sns.boxplot(x = df['y'], y = df[i])
        plt.title(f'{i} by Class')
        plt.xlabel('Class')
        plt.ylabel(i)
        plt.tight_layout()
        plt.savefig(f'{i}_box.png')
        plt.close()

In [5]:
def heatmap(df):
    plt.figure(figsize = (10,8))
    sns.heatmap(df[features].corr(), annot = True, cmap = 'coolwarm', fmt='.2f')
    plt.title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.savefig('feature_corr.png')
    plt.close()

Function to graph y distribution

In [6]:
def class_distribution(y):
    # Get count for each class then create percentage
    unique, count = np.unique(y, return_counts = True)
    percentage = count / count.sum() * 100
    # Label the classes and put count and precentages on each one
    plt.figure()
    bars = plt.bar([f'Class {cls + 1}' for cls in unique], count)
    plt.bar_label(bars, labels = [f'{number} ({perc:.2f}%)' for number,perc in zip(count, percentage)], label_type = 'edge', fontsize = 10)
    
    plt.title('Class distribution of Abalone classes')
    plt.xlabel('Classes')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('class_dist.png')
    plt.close()

Function to split data into 60:40

In [7]:
def split(df): 
    # Split data
    x = df.iloc[:, :-1]
    y =  df.iloc[:, -1]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.4, random_state = 1, stratify = y)
    return x_train, x_test, y_train, y_test

Function to model SGD with varying hidden and lr

In [8]:
def model_sgd(input_dim, hidden, n_layers, lr):
    model = keras.Sequential()
    model.add(layers.Input(shape = (input_dim,)))
    
    if n_layers == 1:
        model.add(layers.Dense(hidden, activation = 'relu'))
    elif n_layers == 2:
        model.add(layers.Dense(hidden, activation = 'relu'))
        model.add(layers.Dense(hidden, activation = 'relu'))
    else:
        raise ValueError('Hidden layers must be either 1 or 2')
        
    model.add(layers.Dense(4, activation = 'softmax'))
    optimise = keras.optimizers.SGD(learning_rate = lr)
    model.compile(loss = 'sparse_categorical_crossentropy', optimizer = optimise, metrics = ['accuracy'])
    return model

Function to model Adam with varying hidden and lr

In [9]:
def model_adam(input_dim, hidden, n_layers, lr):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    if n_layers == 1:
        model.add(layers.Dense(hidden, activation = 'relu'))
    elif n_layers == 2:
        model.add(layers.Dense(hidden, activation = 'relu'))
        model.add(layers.Dense(hidden, activation = 'relu'))
    else:
        raise ValueError('Hidden layers must be either 1 or 2')
        
    model.add(layers.Dense(4, activation = 'softmax'))
    optimise = keras.optimizers.Adam(learning_rate = lr)
    model.compile(loss = 'sparse_categorical_crossentropy', optimizer = optimise, metrics = ['accuracy'])
    return model

Function to fit and evalualte model 

In [10]:
EPOCHS = 120
BATCH_SIZE = 64
PATIENCE = 15

def train(x_train, x_test, y_train, y_test, hidden, n_layers, model_type, lr, initial_weights):
    # Get number of features from x_train
    input_dim = x_train.shape[1]
    if model_type == 'sgd':
        model = model_sgd(input_dim, hidden, n_layers, lr)
    elif model_type == 'adam':
        model = model_adam(input_dim, hidden, n_layers, lr)
    else:
        raise ValueError('model_type must be either sgd or adam')
    # Use same weights for a run
    if initial_weights is not None:
        model.set_weights(initial_weights)
    # Early stopping
    best = keras.callbacks.EarlyStopping(monitor = 'loss', patience = PATIENCE, restore_best_weights = True)
    # Fit model
    model.fit(x_train, y_train, epochs = EPOCHS, batch_size = BATCH_SIZE, verbose = 0, callbacks = [best])
    # Prediction
    train_prob = model.predict(x_train, verbose = 0)
    test_prob = model.predict(x_test, verbose = 0)    
    y_train_pred = np.argmax(train_prob, axis = 1)
    y_test_pred = np.argmax(test_prob, axis = 1)
    
    train_acc  = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)
    return train_acc, test_acc, y_test_pred, test_prob

Function for confusion matrix

In [11]:
def plot_cm(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels = [0,1,2,3])
    fig, ax = plt.subplots(figsize = (5,4))
    
    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(['C1','C2','C3','C4'])
    ax.set_yticklabels(['C1','C2','C3','C4'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    # Add numbers
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(cm[i, j]), ha = 'center', va = 'center', fontsize = 12, color = 'black')
    # Get gridlines
    ax.set_xticks(np.arange(-.5, 4, 1), minor = True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor = True)
    ax.grid(which = 'minor', color='black', linestyle='-', linewidth = 0.5)
    ax.tick_params(which = 'minor', bottom = False, left = False)
    
    plt.tight_layout()
    plt.savefig('cm.png')
    plt.close()

Function for ROC/AUC curve

In [12]:
def roc_multiclass(y_true, y_prob):
    # One hot encode y
    y_true = np.asarray(y_true, dtype=int)
    classes = y_prob.shape[1] 
    y = np.eye(classes)[y_true]
    
    aucs = []
    plt.figure(figsize = (5,4))
    for i in range(classes):
        fpr, tpr, _ = roc_curve(y[:, i], y_prob[:, i])
        auc = roc_auc_score(y[:, i], y_prob[:, i])
        aucs.append(auc)
        plt.plot(fpr, tpr, label = f'Class {i + 1} (AUC = {auc:.2f})')
        
    plt.plot([0,1], [0,1], '--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC (macro AUC = {np.mean(aucs):.2f})')
    plt.legend()
    plt.tight_layout()
    plt.savefig('roc.png')
    plt.close()

Confidence interval

In [13]:
def mean_sd_ci(values):
    values = np.array(values, dtype = float)
    n = len(values)
    mean = values.mean()
    #Sd with division of n-1
    sd = values.std(ddof = 1)
    t = 2.262
    bound = t * sd / np.sqrt(n)
    return mean, sd, bound

In [14]:
def acc_mean(mean, sd, bounds):
    return f'{mean:.2f} ± {sd:.2f} (± {bounds:.2f})'

Function for optimal number of neurons in hidden using SGD

Function to get initial weights

In [15]:
def make_initial_weights(input_dim, hidden, n_layers, run_num):
    tf.keras.utils.set_random_seed(run_num)
    temp = model_sgd(input_dim, hidden, n_layers, lr = 0.01)
    return temp.get_weights()

In [16]:
def neuron(x_train, x_test, y_train, y_test, neurons, lr, n_runs):
    rows = []
    best_hidden, best_acc = None, -1.0
    input_dim = x_train.shape[1]

    acc_tr = {hidden: [] for hidden in neurons}
    acc_te = {hidden: [] for hidden in neurons}

    for i in range(n_runs):
        for hidden in neurons:
            weights = make_initial_weights(input_dim, hidden, n_layers = 1, run_num = i + 1)
            tr, te, _, _ = train(x_train, x_test, y_train, y_test, hidden = hidden, n_layers = 1, model_type = 'sgd', lr = lr, initial_weights = weights)
            acc_tr[hidden].append(tr)
            acc_te[hidden].append(te)

    for hidden in neurons:
        mean_train, sd_train, bound_train = mean_sd_ci(acc_tr[hidden])
        mean_test, sd_test, bound_test = mean_sd_ci(acc_te[hidden])
        rows.append(['1 layer', hidden, 'SGD', lr, acc_mean(mean_train, sd_train, bound_train), acc_mean(mean_test, sd_test, bound_test)])

        if mean_test > best_acc:
            best_hidden, best_acc = hidden, mean_test

    print(f'Best hidden units is: {best_hidden}')
    return best_hidden, rows, best_acc

Function for optimal learning rate using SGD

In [17]:
def lr(x_train, x_test, y_train, y_test, hidden, lrs, n_runs):
    rows = []
    best_lr, best_acc = None, -1.0
    input_dim = x_train.shape[1]
    
    acc_tr = {lr: [] for lr in lrs}
    acc_te = {lr: [] for lr in lrs}
    
    for i in range(n_runs):
        weights = make_initial_weights(input_dim, hidden, n_layers = 1, run_num = i + 1)
        for lr in lrs:
            tr, te, _, _ = train(x_train, x_test, y_train, y_test, hidden = hidden, n_layers = 1, model_type = 'sgd', lr = lr, initial_weights = weights)
            acc_tr[lr].append(tr)
            acc_te[lr].append(te)

    for lr in lrs:
        mean_train, sd_train, bound_train = mean_sd_ci(acc_tr[lr])
        mean_test, sd_test, bound_test = mean_sd_ci(acc_te[lr])
        rows.append(['1 layer', hidden, 'SGD', lr, acc_mean(mean_train, sd_train, bound_train), acc_mean(mean_test, sd_test, bound_test)])
        if mean_test > best_acc:
            best_lr, best_acc = lr, mean_test

    print(f'Best learning rate is: {best_lr}')
    return best_lr, rows, best_acc

Function for comparison of 2 hidden layers using optimal number of hidden neurons and lr using SGD

In [18]:
def two_layers(x_train, x_test, y_train, y_test, hidden, lr, n_runs):
    rows = []
    train_list, test_list = [], []
    input_dim = x_train.shape[1]

    for i in range(n_runs):
        weights = make_initial_weights(input_dim, hidden, n_layers = 2, run_num = i + 1)
        train_acc, test_acc, _, _ = train(x_train, x_test, y_train, y_test, hidden = hidden, n_layers = 2, model_type = 'sgd', lr = lr, initial_weights = weights)
        train_list.append(train_acc)
        test_list.append(test_acc)

    mean_train, sd_train, bound_train = mean_sd_ci(train_list)
    mean_test, sd_test, bound_test  = mean_sd_ci(test_list)
    rows.append(['2 layers', f'{hidden}+{hidden}', 'SGD', lr, acc_mean(mean_train, sd_train, bound_train), acc_mean(mean_test, sd_test, bound_test)])
    
    print(f'Step 4 (2 layers, SGD) mean test acc: {mean_test:.2f}')
    return rows, mean_test

Function for investigate the effect of Adam and SGD on training and test performance

In [19]:
def Adam(x_train, x_test, y_train, y_test, hidden, lr, n_runs):
    rows = []
    train_adam, test_adam = [], []
    input_dim = x_train.shape[1]
    
    for i in range(n_runs):
        weights = make_initial_weights(input_dim, hidden, n_layers = 2, run_num = i + 1)
        train_acc, test_acc, _, _ = train(x_train, x_test, y_train, y_test, hidden = hidden, n_layers = 2, model_type = 'adam', lr = lr, initial_weights = weights)
        train_adam.append(train_acc)
        test_adam.append(test_acc)

    mean_train, sd_train, bound_train = mean_sd_ci(train_adam)
    mean_test, sd_test, bound_test = mean_sd_ci(test_adam)
    rows.append(['2 layers', f'{hidden}+{hidden}', 'Adam', lr, acc_mean(mean_train, sd_train, bound_train), acc_mean(mean_test, sd_test, bound_test)])

    print(f"Step 5 mean test acc Adam: {mean_test:.2f}")
    return rows, mean_test

Execute code

Read data and split

In [20]:
df = read_data()
x_train, x_test, y_train, y_test = split(df)

Analyse and visualise the given datasets by reporting the distribution of classes, distribution of features and any other visualisation you find appropriate.

In [21]:
class_distribution(df['y'])
hist(df)
boxplot(df)
heatmap(df)

count    4177.000000
mean        0.523992
std         0.120093
min         0.075000
25%         0.450000
50%         0.545000
75%         0.615000
max         0.815000
Name: Length, dtype: float64
count    4177.000000
mean        0.407881
std         0.099240
min         0.055000
25%         0.350000
50%         0.425000
75%         0.480000
max         0.650000
Name: Diameter, dtype: float64
count    4177.000000
mean        0.139516
std         0.041827
min         0.000000
25%         0.115000
50%         0.140000
75%         0.165000
max         1.130000
Name: Height, dtype: float64
count    4177.000000
mean        0.828742
std         0.490389
min         0.002000
25%         0.441500
50%         0.799500
75%         1.153000
max         2.825500
Name: Whole_Weight, dtype: float64
count    4177.000000
mean        0.359367
std         0.221963
min         0.001000
25%         0.186000
50%         0.336000
75%         0.502000
max         1.488000
Name: Shucked_Weight, dtype: float64

Develop a dense neural network with one hidden layer. Vary the number of hidden neurons to be 5, 10, 15, and 20 in order to investigate the performance of the model using Stochastic Gradient Descent (SGD). Determine the optimal number of neurons in the hidden layer from the range of values considered.

In [22]:
best_hidden, rows2, step2_best = neuron(x_train, x_test, y_train, y_test, neurons = [5, 10, 15, 20], lr = 0.01, n_runs = 10)

Best hidden units is: 15


Investigate the effect of learning rate (using SGD) for the selected dataset (using the optimal number of hidden neurons).

In [23]:
best_lr, rows3, step3_best = lr(x_train, x_test, y_train, y_test, hidden = best_hidden, lrs = [0.0001, 0.001, 0.01, 0.1], n_runs = 10)

Best learning rate is: 0.1


Investigate the effect on a different number of hidden layers:  Now modify the model by adding another hidden layer. Use the optimal number of hidden neurons from Step 2 for both the layers and the optimal learning rate from Step 3. Investigate the effect of this change in the number of hidden layers (using SGD). 

In [24]:
rows4, sgd_mean = two_layers(x_train, x_test, y_train, y_test, hidden = best_hidden, lr = best_lr, n_runs = 10)

Step 4 (2 layers, SGD) mean test acc: 0.70


Investigate the effect of Adam and SGD on training and test performance. 

In [25]:
rows5, adam_mean = Adam(x_train, x_test, y_train, y_test, hidden = best_hidden, lr = best_lr, n_runs = 10)

Step 5 mean test acc Adam: 0.72


Create our summary table

In [26]:
summary = pd.DataFrame(rows2 + rows3 + rows4 + rows5, columns=['Layers', 'Hidden Units', 'Optimizer', 'Learning Rate', 'Train Acc', 'Test Acc'])
summary.to_csv('summary_results.csv', index = False)
print(summary)

     Layers Hidden Units Optimizer  Learning Rate             Train Acc  \
0   1 layer            5       SGD         0.0100  0.66 ± 0.00 (± 0.00)   
1   1 layer           10       SGD         0.0100  0.66 ± 0.01 (± 0.00)   
2   1 layer           15       SGD         0.0100  0.66 ± 0.00 (± 0.00)   
3   1 layer           20       SGD         0.0100  0.66 ± 0.00 (± 0.00)   
4   1 layer           15       SGD         0.0001  0.51 ± 0.15 (± 0.10)   
5   1 layer           15       SGD         0.0010  0.65 ± 0.02 (± 0.01)   
6   1 layer           15       SGD         0.0100  0.66 ± 0.00 (± 0.00)   
7   1 layer           15       SGD         0.1000  0.71 ± 0.02 (± 0.01)   
8  2 layers        15+15       SGD         0.1000  0.70 ± 0.04 (± 0.03)   
9  2 layers        15+15      Adam         0.1000  0.71 ± 0.01 (± 0.01)   

               Test Acc  
0  0.67 ± 0.00 (± 0.00)  
1  0.67 ± 0.00 (± 0.00)  
2  0.67 ± 0.00 (± 0.00)  
3  0.67 ± 0.00 (± 0.00)  
4  0.51 ± 0.14 (± 0.10)  
5  0.65 ± 0.02 (± 

Take the final optimal model among all the above cases and show the confusion matrix and ROC/AUC curve for different classes of the multi-class problem.

In [27]:
if adam_mean > sgd_mean:
    final_model = 'adam'
else:
    final_model = 'sgd'

train_acc, test_acc, y_pred, y_prob = train(x_train, x_test, y_train, y_test,hidden = best_hidden, n_layers = 2, model_type = final_model, lr = best_lr, initial_weights = None)

plot_cm(y_test, y_pred)
roc_multiclass(y_test, y_prob)

print(f'Final model: {final_model.upper()}, lr = {best_lr}, hidden = {best_hidden}, 'f'train_acc = {train_acc:.2f}, test_acc = {test_acc:.2f}')

Final model: ADAM, lr = 0.1, hidden = 15, train_acc = 0.70, test_acc = 0.71
